---
## Section 5: Failure Modes — Hallucination & Reasoning Gaps

### The 3 Failure Modes You Must Design For

| Failure Mode | Mechanism | Looks Like | Mitigation |
|---|---|---|---|
| **Hallucination** | Model generates plausible-looking content that doesn't exist | Confident wrong citation, invented statistic | Retrieval grounding + source mapping |
| **Reasoning Gap** | Model reproduces *patterns* of computation, not actual computation | Wrong math that looks right | Tool use (code execution for arithmetic) |
| **Confident Wrong Output** | RLHF trains against hedging; model sounds sure even when wrong | Definitive answer to ambiguous question | Explicit uncertainty elicitation |

**The critical insight:** These outputs look *identical* to correct outputs. The model doesn't signal that it's wrong.

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Hallucination — Citation Fabrication
#
# The model is asked for a specific academic citation.
# When it doesn't exist in training data, it generates a
# 'citation-shaped' token sequence — looks real, isn't.
#
# WHY: It's not malfunctioning. It's generating text that fits
# the pattern of 'what a citation looks like here'.
# ─────────────────────────────────────────────────────────────

def ask_for_citation(topic: str, temperature: float = 0.7) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temperature,
        messages=[{
            "role": "user",
            "content": f"Provide a specific academic paper citation (APA format) about: {topic}"
        }]
    )
    return response.choices[0].message.content


# Topics where hallucination risk is high (niche/recent/specific)
niche_topics = [
    "attention degradation in the middle of long transformer contexts (2023)",
    "tokenization effects on structured output compliance in GPT-4",
]

print("HALLUCINATION DEMO — Do not trust these citations without verification!")
print("=" * 70)
for topic in niche_topics:
    print(f"\nTopic: {topic}")
    citation = ask_for_citation(topic)
    print(f"Model output: {citation}")
    print("\nACTION REQUIRED: Search for this paper on Google Scholar or Semantic Scholar.")
    print("   If it doesn't exist — this is a hallucination. It looked real.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Reasoning Gap — Arithmetic Failures
#
# The model generates PATTERNS of computation, not actual computation.
# Simple arithmetic in isolation: usually correct (common training pattern).
# Multi-step or unusual arithmetic: failure risk increases significantly.
#
# MITIGATION: Never use an LLM for arithmetic. Call a calculator.
# ─────────────────────────────────────────────────────────────

math_problems = [
    # (problem_text, correct_answer)
    ("What is 17 * 43?", 731),
    ("What is 2^10 + 2^8 - 100?", 1024 + 256 - 100),  # = 1180
    ("A vendor charges $0.0001 per token. A document has 45,823 tokens. What is the cost in dollars?", 45823 * 0.0001),
    # Compound multi-step problem — higher failure probability
    ("If a model processes 1,247 documents per hour, each averaging 3,891 tokens, "
     "and the rate limit is 5,000,000 tokens per minute, will it hit the rate limit? "
     "Show your calculation.", None),  # No single correct answer to check — observe reasoning
]

def ask_math(problem: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,  # Deterministic — removes sampling as a variable
        messages=[{"role": "user", "content": problem}]
    )
    return response.choices[0].message.content


print("REASONING GAP DEMO — Arithmetic Failures")
print("=" * 60)
for problem, correct in math_problems:
    answer = ask_math(problem)
    print(f"\nProblem: {problem}")
    print(f"Model: {answer.strip()[:200]}")
    if correct is not None:
        print(f"Correct answer: {correct}")
        # Very rough check — look for the number in the output
        is_correct = str(int(correct)) in answer.replace(",", "").replace(" ", "")
        print(f"Assessment: {'Correct' if is_correct else 'WRONG — do not trust LLM arithmetic'}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Eliciting Uncertainty
#
# By default, models are trained to sound confident (hedging
# is often rated poorly in RLHF). But you can explicitly
# prompt for calibrated uncertainty.
#
# Compare: confident prompt vs. calibrated prompt
# ─────────────────────────────────────────────────────────────

# A question where the model may not have reliable knowledge
question = "What is the exact token limit for Claude 3 Opus as of today?"

confident_prompt = question  # No hedging instruction — default confident behavior

calibrated_prompt = f"""{question}

If you are not certain, say so explicitly. 
Rate your confidence: High / Medium / Low.
Explain what you are uncertain about."""

print("=" * 60)
print("PROMPT TYPE: Default (no uncertainty instruction)")
resp_confident = ask_math(confident_prompt)
print(resp_confident)

print("\n" + "=" * 60)
print("PROMPT TYPE: Calibrated (explicit uncertainty elicitation)")
resp_calibrated = ask_math(calibrated_prompt)
print(resp_calibrated)

print("\n💡 Takeaway: The calibrated prompt surfaces uncertainty that was hidden.")
print("   Design review processes around what the model is uncertain about.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# TODO 5 — Build a Basic Output Inspector (10 min)
#
# This is the most production-relevant exercise in the notebook.
#
# TASK:
#   Build a simple function that programmatically inspects
#   a batch of LLM outputs and logs:
#     (a) Whether each output is valid JSON
#     (b) Whether required fields are present
#     (c) Whether values are within expected ranges
#     (d) A summary failure report
#
# This is the foundation of any production monitoring system.
# ─────────────────────────────────────────────────────────────

# The prompt we're monitoring
EXTRACTION_PROMPT = """
Extract contract metadata and return ONLY valid JSON:
{"vendor": string, "sla_percent": number, "payment_days": integer, "termination_days": integer}

Contract: {contract_text}
"""

# Sample contracts to test against
test_contracts = [
    "DataStream Inc will maintain 99.5% uptime. Invoices due in 30 days. 90 days notice to terminate.",
    "CloudBase agrees to 99.9% SLA, net-60 payment, 45 day termination notice.",
    "Vertex Systems provides infrastructure. Contract renewable annually.",  # Missing fields — edge case
    "SLA: 99.95%. Pay: 15 days. Exit: 30 days. Vendor: NimbusAI",
]

# Required fields and their expected types
REQUIRED_FIELDS = {
    "vendor": str,
    "sla_percent": (int, float),
    "payment_days": int,
    "termination_days": int,
}

def inspect_output(raw_output: str, required_fields: dict) -> dict:
    """
    Inspects a single LLM output.
    
    Returns a report dict with:
      - is_valid_json: bool
      - missing_fields: list of missing field names
      - type_errors: list of fields with wrong types
      - parsed: the parsed dict (or None if invalid JSON)
    """
    report = {
        "raw": raw_output,
        "is_valid_json": False,
        "missing_fields": [],
        "type_errors": [],
        "parsed": None,
    }
    
    # Step 1: Check JSON validity
    try:
        parsed = json.loads(raw_output)
        report["is_valid_json"] = True
        report["parsed"] = parsed
    except json.JSONDecodeError:
        return report  # Can't continue if not valid JSON
    
    # Step 2: Check for required fields
    for field, expected_type in required_fields.items():
        if field not in parsed:
            report["missing_fields"].append(field)
        elif not isinstance(parsed[field], expected_type):
            report["type_errors"].append(f"{field}: expected {expected_type}, got {type(parsed[field])}")
    
    return report


# --- YOUR CODE HERE ---
# Task: Run each contract through the model, inspect the output,
# collect all reports, and print a summary failure report.

# all_reports = []
# for contract in test_contracts:
#     prompt = EXTRACTION_PROMPT.format(contract_text=contract)
#     raw = ask_math(prompt)  # Reuse the ask function
#     report = inspect_output(raw, REQUIRED_FIELDS)
#     all_reports.append({"contract": contract[:60], "report": report})
#
# # Print summary
# print("\nOUTPUT INSPECTION REPORT")
# print("=" * 60)
# for item in all_reports:
#     r = item["report"]
#     status = "OK" if r["is_valid_json"] and not r["missing_fields"] and not r["type_errors"] else "NOT_OK"
#     print(f"{status} Contract: {item['contract']}")
#     if not r["is_valid_json"]:
#         print(f"   Invalid JSON")
#     if r["missing_fields"]:
#         print(f"   Missing: {r['missing_fields']}")
#     if r["type_errors"]:
#         print(f"   Type errors: {r['type_errors']}")